In [1]:
from pathlib import Path
import os
import json
import time
from groq import Groq,RateLimitError
import random
import pandas as pd

from sqlalchemy import text
from sqlalchemy import create_engine
from sqlalchemy.orm import Session


from deepeval.synthesizer import Synthesizer
from deepeval.models import DeepEvalBaseLLM
from deepeval.dataset import EvaluationDataset, Golden

In [2]:
import torch
print(torch.cuda.is_available())

True


In [3]:
!nvidia-smi

Tue Mar 17 01:38:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.83                 Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   49C    P8              1W /   30W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL, echo=False)

In [5]:
shrunk_dataset_path = Path(r"C:\Uni\4th Year\GP\ECHO\data\chatbot\outputs\evaluation_data\shrunk_dataset_132.csv")

In [7]:
import pandas as pd

def fetch_all_entities_with_chunks(csv_path: str):
    """
    Fetch all pharaohs and landmarks with their text chunks,
    filtered to only include contexts that appear in the CSV file.
    Entities with no matching contexts are excluded entirely.
    """
    
    # Load the CSV and extract valid contexts as a set for O(1) lookup
    df = pd.read_csv(csv_path)
    valid_contexts = set(df["context"].dropna().str.strip())
    
    entities_data = []
    
    with Session(engine) as session:
        # Fetch all pharaohs with their chunks
        pharaohs_query = """
        SELECT 
            p.name,
            pt.text_chunk
        FROM pharaohs p
        JOIN pharaohs_texts pt ON p.id = pt.pharaoh_id
        ORDER BY p.name
        """
        pharaoh_results = session.execute(text(pharaohs_query)).fetchall()
        
        # Group chunks by pharaoh, filter by length AND CSV contexts
        pharaoh_chunks = {}
        for name, chunk in pharaoh_results:
            if chunk and len(chunk.strip()) >= 5 and chunk.strip() in valid_contexts:
                if name not in pharaoh_chunks:
                    pharaoh_chunks[name] = []
                pharaoh_chunks[name].append(chunk.strip())
        
        for name, chunks in pharaoh_chunks.items():
            if chunks:
                entities_data.append({
                    "entity_type": "pharaoh",
                    "entity_name": name,
                    "contexts": chunks
                })

        # Fetch all landmarks with their chunks
        landmarks_query = """
        SELECT 
            l.name,
            lt.text_chunk
        FROM landmarks l
        JOIN landmarks_texts lt ON l.id = lt.landmark_id
        ORDER BY l.name
        """
        landmark_results = session.execute(text(landmarks_query)).fetchall()
        
        # Group chunks by landmark, filter by length AND CSV contexts
        landmark_chunks = {}
        for name, chunk in landmark_results:
            if chunk and len(chunk.strip()) >= 5 and chunk.strip() in valid_contexts:
                if name not in landmark_chunks:
                    landmark_chunks[name] = []
                landmark_chunks[name].append(chunk.strip())
        
        for name, chunks in landmark_chunks.items():
            if chunks:
                entities_data.append({
                    "entity_type": "landmark",
                    "entity_name": name,
                    "contexts": chunks
                })
    
    return entities_data


# Usage
x = fetch_all_entities_with_chunks(shrunk_dataset_path)
x

[{'entity_type': 'pharaoh',
  'entity_name': 'Achoris',
  'contexts': ['Domestically, Hakoris’s eighth regnal year was marked by a significant internal rebellion that was suppressed by General Nakhtnebef. Despite continued Persian aggression, Hakoris maintained Egyptian sovereignty and ensured relative peace and prosperity within his borders. His reign coincided with a period of complex international relations, including peace between Persia and Sparta but intermittent warfare among Greek city-states. Egypt under Hakoris emerged as a significant regional power allied with Greek and Cypriot interests, maintaining stability and independence during a time of shifting alliances across the eastern Mediterranean.']},
 {'entity_type': 'pharaoh',
  'entity_name': 'Ahmose I',
  'contexts': ['Ahmose’s military career began with the war of liberation against the Hyksos. He crushed the Hyksos’ allies in Middle Egypt, advanced down the Nile River, and captured Memphis, the traditional capital of Eg

In [10]:
class GroqKeyManager:
    def __init__(self, keys):
        self.keys = keys
        self.current_index = 0
        self.clients = [Groq(api_key=k) for k in keys]

    def get_client(self):
        return self.clients[self.current_index]

    def rotate_key(self):
        self.current_index = (self.current_index + 1) % len(self.keys)
        print(f"🔄 Switched to API Key {self.current_index + 1}/{len(self.keys)}")
        # A 2-second breather helps the new key's "token bucket" stabilize
        time.sleep(2)

def generate_deepeval_benchmark(entities_data, questions_per_entity=3):
    keys = [
        os.getenv("GROQ_API_KEY1"),
        os.getenv("GROQ_API_KEY2"),
        os.getenv("GROQ_API_KEY3"),
        os.getenv("GROQ_API_KEY4"),
        os.getenv("GROQ_API_KEY5"),
        os.getenv("GROQ_API_KEY6"), 
        os.getenv("GROQ_API_KEY7")
    ]
    manager = GroqKeyManager([k for k in keys if k]) 
    deepeval_goldens = []
    
    print(f"\n🚀 Starting Synthesis with {len(manager.keys)} API keys...")

    for i, entity in enumerate(entities_data, 1):
        name = entity['entity_name']
        valid_chunks = [c for c in entity['contexts'] if len(c.strip()) > 5]
        if not valid_chunks: continue

        print(f"[{i}/{len(entities_data)}] {name}...")
        
        chunks_to_use = random.sample(valid_chunks, min(questions_per_entity, len(valid_chunks)))

        for chunk in chunks_to_use:
            
            prompt = f"""

            You are generating a test case for an AI persona system where {name} speaks in first person.

            TEXT SEGMENT:
            {chunk}

            TASK:
            Generate 1 question and 1 structured answer from the TEXT SEGMENT only.

            STRICT PERSPECTIVE & STYLE:
            1. **The Question**: Must be directed to {name} in the second person.
            2. **The Answer (First Person)**: Must be written in the FIRST PERSON as {name}.
            (e.g., "I built...", "My reign was...", "I am located...", etc.)
            3. **Accuracy**: Every historical fact must come ONLY from the TEXT SEGMENT.

            MANDATORY ANSWER STRUCTURE (strictly follow this 2-part structure):
            - **Paragraph 1**: Directly answer the question with concrete facts only
            (dates, names, events, locations). No elaboration yet.
            - **Remaining Paragraphs**: Expand on the WHY and HOW in an immersive, first-person voice. 

            EXAMPLE STRUCTURE:
            {{
            "question": "...",
            "answer": "I [direct factual answer with dates/names/events].\\n\\n[Poetic elaboration 
                        expanding on why and how, staying in character as {name}, grounded in 
                        the facts above.]"
            }}

            FORMAT (Strict JSON):
            {{
            "question": "...",
            "answer": "..."
            }}
"""

            success = False
            # Allow each key a chance to try this specific chunk
            attempts_with_current_chunk = 0
            
            
            while not success and attempts_with_current_chunk < len(manager.keys) * 2:
                try:
                    client = manager.get_client()
                    response = client.chat.completions.create(
                        model="openai/gpt-oss-120b",
                        messages=[{"role": "user", "content": prompt}],
                        temperature=0.7,
                        max_tokens=4096,
                        response_format={"type": "json_object"},
                        extra_body={"reasoning_effort": "medium", "reasoning_format": "hidden"}
                    )
                    
                    qa = json.loads(response.choices[0].message.content)
                    
                    golden = Golden(
                        input=qa["question"],
                        expected_output=qa["answer"],
                        context=[chunk],
                        additional_metadata={
                            "entity_name": name, 
                            "entity_type": entity['entity_type']
                        }
                    )
                    deepeval_goldens.append(golden)
                    success = True
                    
                    # ⏱️ THE SWEET SPOT: 1.2 seconds 
                    # This prevents 401/429 errors by keeping RPM < 60 per key
                    time.sleep(1.3) 

                except RateLimitError:
                    print(f"  ⚠️ Rate Limit (TPM/RPM) on Key {manager.current_index + 1}. Rotating...")
                    manager.rotate_key()
                    attempts_with_current_chunk += 1
                    
                    # If we've rotated back to the first key, wait a bit longer
                    if manager.current_index == 0:
                        print("  ⏳ All keys exhausted. Short 10s cooldown...")
                        time.sleep(10)
                        
                except Exception as e:
                    print(f"  ✗ Unexpected Error: {e}")
                    break 

    # --- Save Process ---
    dataset = EvaluationDataset(goldens=deepeval_goldens)
    output_dir = Path(r"C:\Uni\4th Year\GP\ECHO\data\chatbot\outputs\new_evaluation_data")
    output_dir.mkdir(exist_ok=True)
    
    dataset.save_as(file_type="json", directory=str(output_dir))
    print(f"\n✅ Benchmark Complete! {len(deepeval_goldens)} test cases saved.")
    
    return dataset

In [11]:
entities_data = fetch_all_entities_with_chunks(shrunk_dataset_path)
dataset = generate_deepeval_benchmark(entities_data, questions_per_entity=1)


🚀 Starting Synthesis with 7 API keys...
[1/132] Achoris...
[2/132] Ahmose I...
[3/132] Akhenaton...
[4/132] Alexander The Great...
[5/132] Amasis II...
[6/132] Amenemhat I...
[7/132] Amenemhat III...
[8/132] Amenhotep II...
[9/132] Amenhotep III...
[10/132] Amenirdis (Daughter of Kashta)...
[11/132] Amun (God)...
[12/132] Amun-Ra (God)...
[13/132] Anath Goddess...
[14/132] Ankhsenamun...
[15/132] Anubis (God)...
[16/132] Arsinoe III (Queen, daughter of Ptolemy I and wife of Ptolemy II)...
[17/132] Arsinoe II (Queen, sister and wife of Ptolemy IV)...
[18/132] Bat Goddess...
  ⚠️ Rate Limit (TPM/RPM) on Key 1. Rotating...
🔄 Switched to API Key 2/7
[19/132] Cleopatra VII Philopator...
[20/132] Djoser...
[21/132] Hathor (Goddess)...
[22/132] Hatshepsut...
[23/132] Hauron God...
[24/132] Horemheb...
[25/132] Hor I...
[26/132] Horus (God)...
[27/132] Isis (Goddess)...
[28/132] Isis (mother of Thutmose III)...
[29/132] Khafre...
[30/132] Khamerernebty II...
[31/132] Khasekhem...
[32/132] Kho

In [12]:
import pandas as pd
from pathlib import Path

# 1. Extract the data from the 'goldens' list inside the dataset
data_list = []
for golden in dataset.goldens:
    row = {
        "input": golden.input,
        "expected_output": golden.expected_output,
        # Join the context list into one string for CSV readability
        "context": " ".join(golden.context) if isinstance(golden.context, list) else golden.context,
        # Flatten the metadata directly
        "entity_name": golden.additional_metadata.get("entity_name"),
        "entity_type": golden.additional_metadata.get("entity_type")
    }
    data_list.append(row)

# 2. Create the DataFrame
df = pd.DataFrame(data_list)

# 3. Save to your evaluation folder
output_dir = Path(r"C:\Uni\4th Year\GP\ECHO\data\chatbot\outputs\new_evaluation_data")
output_dir.mkdir(exist_ok=True)
csv_path = output_dir / "synthetic_dataset.csv"

# Using utf-8-sig helps Excel handle special historical characters/dates correctly
df.to_csv(csv_path, index=False, encoding='utf-8-sig')


# Display first few rows
df.head()

,input,expected_output,context,entity_name,entity_type
0,You faced a significant internal rebellion in ...,I suppressed the rebellion in my eighth year w...,"Domestically, Hakoris’s eighth regnal year was...",Achoris,pharaoh
1,You led several campaigns during your reign; c...,I began my military career with the war of lib...,Ahmose’s military career began with the war of...,Ahmose I,pharaoh
2,Why did you imprison Abdiashirta for a year an...,I imprisoned Abdiashirta for a year and releas...,". However, Akhenaten did intervene selectively...",Akhenaton,pharaoh
3,You are said to have started the fire that des...,I started the fire that destroyed the main pal...,According to the ancient historian Diodorus Si...,Alexander The Great,pharaoh
4,"Where were you buried, and what happened to yo...","I was buried at Sais, as shabtis and fragments...",". Though the tomb has not been found, shabtis ...",Amasis II,pharaoh


In [13]:
df.tail()

,input,expected_output,context,entity_name,entity_type
127,You display decorative elements that symbolize...,I have columns with Hathoric capitals that ser...,"The columns of the mammisi, with Hathoric capi...",The Roman Mammisi in Dendera,landmark
128,What are the features of your Second Court and...,I have a second set of pylon gateways with sma...,A second set of pylon gateways with smaller en...,The Small Aten Temple,landmark
129,When were you rediscovered and what happened t...,I was rediscovered in 1975 after having been i...,The location of the tomb was not always known....,Tomb of Horemheb in Saqqara,landmark
130,"What circumstances led to your untimely death,...",I drowned when my boat overturned while sailin...,Isidora was a wealthy and beautiful young woma...,Tomb of Isadora,landmark
131,What do the registers on the left side of your...,I show three scenes on the left side: the lowe...,. On the left side the lower register represen...,Tomb of Petosiris,landmark
